# LangGraph — Complex Agent

**Framework:** LangGraph  
**Level:** 03 — Complex  
**Model:** `gemini-2.0-flash`

### What's New vs Intermediate
| Feature | Intermediate | Complex |
|---|---|---|
| Graph nodes | 2 (llm, tools) | **6** (planner, researcher, tools, drafter, critic, reviser) |
| State fields | messages + 2 extras | **messages + plan + draft + critique + score + attempt** |
| Routing | `tools_condition` only | **Custom conditional edges** based on state fields |
| Reflexion | None | **Critic node + conditional retry edge** |
| Streaming | `.invoke()` | **`graph.stream()` — see every node transition** |

## Concept: Multi-Node Reflexion Graph

```
START
  │
  ▼
[planner_node]          writes plan → to state.plan
  │
  ▼
[researcher_node]       calls LLM with tools → may call tools
  │
  ├──(tool_calls?)──▶ [tools_node] ──▶ back to researcher
  │
  └──(no tool_calls)──▶ [drafter_node]    synthesizes data → state.draft_report
                              │
                              ▼
                        [critic_node]     scores draft → state.quality_score
                              │
                  ┌───────────┴───────────┐
              score >= 7              score < 7
              OR attempt >= 2         AND attempt < 2
                  │                       │
                  ▼                       ▼
                 END              [reviser_node]  → rewrites draft
                                         │
                                         └──▶ back to [critic_node]
```

**Why LangGraph for reflexion?**
- The retry loop is a **first-class graph edge** — you can see it, modify it, test it
- Max retries are enforced by the routing function, not by LLM instruction
- `graph.stream()` shows you every node transition — full observability
- Each node is a pure function — easy to unit-test in isolation

## Setup

In [ ]:
import os
from typing import Annotated, Optional
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langgraph.graph import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## Tools

In [ ]:
@tool
def get_weather(city: str) -> dict:
    """Return weather data and weather score for a city."""
    data = {
        "london":    {"condition": "Cloudy",       "temp_c": 12, "score": 5},
        "new york":  {"condition": "Sunny",         "temp_c": 22, "score": 8},
        "bangalore": {"condition": "Rainy",         "temp_c": 25, "score": 6},
        "tokyo":     {"condition": "Clear",         "temp_c": 18, "score": 9},
        "paris":     {"condition": "Partly Cloudy", "temp_c": 16, "score": 7},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No data."}


@tool
def get_time(city: str) -> dict:
    """Return local time for a city."""
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    return {"city": city, "local_time": times.get(city.lower(), "Unknown")}


@tool
def get_travel_advisory(city: str) -> dict:
    """Return safety advisory and safety score for a city."""
    data = {
        "london":    {"level": "Low",    "notes": "Routine precautions.",      "safety_score": 8},
        "new york":  {"level": "Low",    "notes": "Normal precautions.",       "safety_score": 7},
        "bangalore": {"level": "Medium", "notes": "Monsoon affects transport.", "safety_score": 6},
        "tokyo":     {"level": "Low",    "notes": "Very safe city.",            "safety_score": 10},
        "paris":     {"level": "Low",    "notes": "Alert in crowded spots.",    "safety_score": 8},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No advisory."}


tools = [get_weather, get_time, get_travel_advisory]
print(f"{len(tools)} tools defined")

## Rich State

Every field tracks a specific stage of the workflow. Nodes read from and write to this state.

In [ ]:
class PlannerState(TypedDict):
    messages:      Annotated[list, add_messages]  # conversation + tool call history
    plan:          Optional[str]    # written by planner_node
    draft_report:  Optional[str]    # written by drafter_node, updated by reviser_node
    critique:      Optional[str]    # written by critic_node
    quality_score: int              # written by critic_node
    attempt:       int              # incremented by critic_node
    goal:          str              # the original user goal

print("State fields:", list(PlannerState.__annotations__.keys()))

## 6 Nodes

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# --- Node 1: Planner ---
def planner_node(state: PlannerState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Create a numbered research plan. Each line: 'N. Research [city]: weather, time, advisory'"),
        ("human", "{goal}"),
    ])
    plan = (prompt | llm | StrOutputParser()).invoke({"goal": state["goal"]})
    print(f"  [planner] Plan written ({len(plan)} chars)")
    return {"plan": plan, "messages": [AIMessage(content=f"Plan:\n{plan}")]}


# --- Node 2: Researcher (LLM with tools) ---
RESEARCH_SYS = SystemMessage(content=(
    "You are a travel researcher. Call get_weather, get_time, and get_travel_advisory "
    "for EVERY city mentioned in the plan. Do not skip any city."
))

def researcher_node(state: PlannerState) -> dict:
    instruction = f"Execute this research plan:\n{state['plan']}\n\nCall tools for all cities."
    messages = [RESEARCH_SYS, HumanMessage(content=instruction)] + state["messages"][-6:]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# --- Node 3: Tools (prebuilt) ---
tools_node = ToolNode(tools)


# --- Node 4: Drafter ---
def drafter_node(state: PlannerState) -> dict:
    tool_msgs = [m for m in state["messages"] if hasattr(m, "type") and m.type == "tool"]
    data_summary = "\n".join(str(m.content)[:200] for m in tool_msgs[-12:])
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Write a travel comparison report. Include: ranked cities list, weather scores, "
         "safety scores, local times, and a final recommendation. Minimum 300 words."),
        ("human", f"Research data:\n{data_summary}\n\nWrite the comparison report:"),
    ])
    draft = (prompt | llm | StrOutputParser()).invoke({})
    print(f"  [drafter] Draft written ({len(draft)} chars)")
    return {"draft_report": draft, "messages": [AIMessage(content="Draft report written.")]}


# --- Node 5: Critic ---
def critic_node(state: PlannerState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Score this travel report 1-10. Check: has rankings? specific numbers? safety info? 200+ chars?\n"
         "Respond ONLY: SCORE:[number] VERDICT:[PASS/REVISE] ISSUES:[text or None]"),
        ("human", f"Report:\n{state['draft_report']}"),
    ])
    critique = (prompt | llm | StrOutputParser()).invoke({})

    score = 5
    for part in critique.split():
        if part.startswith("SCORE:"):
            try:
                score = int(part.split(":")[1])
            except ValueError:
                pass

    attempt = state.get("attempt", 0) + 1
    print(f"  [critic] Score: {score}/10, attempt {attempt}, verdict: {'PASS' if score >= 7 else 'REVISE'}")
    return {
        "critique": critique,
        "quality_score": score,
        "attempt": attempt,
        "messages": [AIMessage(content=f"Critique: {critique}")],
    }


# --- Node 6: Reviser ---
def reviser_node(state: PlannerState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Improve this travel report. Fix the issues. Keep all existing data."),
        ("human", f"Original:\n{state['draft_report']}\n\nCritique:\n{state['critique']}\n\nImproved:"),
    ])
    revised = (prompt | llm | StrOutputParser()).invoke({})
    print(f"  [reviser] Report revised ({len(revised)} chars)")
    return {
        "draft_report": revised,
        "messages": [AIMessage(content=f"Report revised (attempt {state['attempt']})")],
    }


print("All 6 nodes defined")

## Routing Functions

These are the **decision functions** that LangGraph calls at conditional edges.

In [ ]:
def route_after_research(state: PlannerState) -> str:
    """After researcher: route to tools if tool_calls present, else to drafter."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "drafter"


def route_after_critic(state: PlannerState) -> str:
    """After critic: END if quality passes or max retries reached; else revise."""
    if state["quality_score"] >= 7 or state.get("attempt", 0) >= 2:
        return END
    return "reviser"


print("Routing functions defined")

## Build and Compile the Graph

In [ ]:
builder = StateGraph(PlannerState)

# Add all nodes
builder.add_node("planner",    planner_node)
builder.add_node("researcher", researcher_node)
builder.add_node("tools",      tools_node)
builder.add_node("drafter",    drafter_node)
builder.add_node("critic",     critic_node)
builder.add_node("reviser",    reviser_node)

# Wire the edges
builder.set_entry_point("planner")
builder.add_edge("planner", "researcher")
builder.add_conditional_edges("researcher", route_after_research,
                               {"tools": "tools", "drafter": "drafter"})
builder.add_edge("tools", "researcher")
builder.add_edge("drafter", "critic")
builder.add_conditional_edges("critic", route_after_critic,
                               {"reviser": "reviser", END: END})
builder.add_edge("reviser", "critic")  # ← the reflexion loop

graph = builder.compile()
print("Graph compiled")

# Visualize the graph
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Visualization skipped — install graphviz to enable)")

## Run with `graph.stream()` — See Every Node Transition

In [ ]:
def run(goal: str) -> str:
    print(f"Goal: {goal}\n")
    final_state = None

    # stream_mode="updates" yields {node_name: state_update} for each step
    for step in graph.stream(
        {"messages": [], "goal": goal, "attempt": 0, "quality_score": 0,
         "plan": None, "draft_report": None, "critique": None},
        stream_mode="updates",
    ):
        for node_name, update in step.items():
            msgs = update.get("messages", [])
            if msgs:
                last = msgs[-1]
                content = last.content if isinstance(last.content, str) else str(last.content)
                print(f"  [{node_name}] → {content[:80].replace(chr(10), ' ')}")

    # Get final state
    final = graph.invoke(
        {"messages": [], "goal": goal, "attempt": 0, "quality_score": 0,
         "plan": None, "draft_report": None, "critique": None},
    )
    return final["draft_report"]


goal = "Compare Tokyo, Paris, and Bangalore. Rank by best weather and safety."
print("--- Node trace ---")
# Run stream for trace
for step in graph.stream(
    {"messages": [], "goal": goal, "attempt": 0, "quality_score": 0,
     "plan": None, "draft_report": None, "critique": None},
    stream_mode="updates",
):
    for node_name in step.keys():
        print(f"  ✓ {node_name}")

In [ ]:
# Full run — get the final report
final = graph.invoke(
    {"messages": [], "goal": goal, "attempt": 0, "quality_score": 0,
     "plan": None, "draft_report": None, "critique": None},
)

print(f"Quality score: {final['quality_score']}/10")
print(f"Attempts: {final['attempt']}")
print(f"\nFinal report ({len(final['draft_report'])} chars):\n")
print(final["draft_report"])

In [ ]:
# Show the execution plan that was generated
print("Generated plan:")
print(final["plan"])
print(f"\nFinal critique: {final['critique']}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| 6 focused nodes | Each does one thing — testable, replaceable, inspectable |
| `add_conditional_edges` with dict | Explicit routing table — you see all possible transitions |
| `reviser → critic` edge | The reflexion loop is a **named graph edge**, not hidden Python logic |
| `route_after_critic` function | Max retries enforced in code — LLM can't loop forever |
| `graph.stream(stream_mode="updates")` | Node-level streaming — each yielded item is `{node_name: updates}` |
| Rich state tracks all stages | `draft_report`, `critique`, `quality_score` all inspectable after run |

### All 4 Frameworks — Complex Level Comparison

| Dimension | ADK | LangChain | LangGraph | CrewAI |
|---|---|---|---|---|
| Planning | In instruction | Planner chain | `planner_node` | `planner` agent |
| Reflexion | `score_report` tool | Python loop | Critic node + edge | Critic agent + task |
| Retry control | LLM decides | `max_retries` param | `route_after_critic` | `max_iter` on agent |
| Streaming | Async event stream | `.stream()` | `graph.stream()` | Verbose output |
| Observability | Medium | Medium | **High** | Low |

### Next Steps
- Compare with [CrewAI 03-Complex →](../../CrewAI/03-complex/agent.ipynb)
- Try adding a new node (e.g., a `human_review` node with `interrupt_before`)
- Swap `MemorySaver` back in and compare checkpointed vs. non-checkpointed behavior